# 🌐 iTech Academy — Browser Agent Demo

## An AI Agent that opens a browser, reads the web, and answers your question!

```
You ask a question
      ↓
  🔍 Agent searches DuckDuckGo
      ↓
  🌐 Browser opens the top result  ← you can SEE it!
      ↓
  📄 Agent reads the page content
      ↓
  🤖 LLM answers from what it read
      ↓
  ✅ Answer!
```

**Stack:** Groq (free LLM) + LangGraph + DuckDuckGo + requests + BeautifulSoup

In [10]:
!pip install -q langchain langchain-community langchain-groq langgraph ddgs requests beautifulsoup4

In [ ]:
import os
import webbrowser
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS

from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent

os.environ['GROQ_API_KEY'] = ''

llm = ChatGroq(model='llama-3.1-8b-instant', temperature=0)
print('✅ Ready!')

✅ Ready!


---
## Tools

| Tool | What it does |
|------|--------------|
| `search_web` | Searches DuckDuckGo, returns top URLs |
| `open_and_read` | **Opens browser** + scrapes page text |

In [12]:
@tool
def search_web(query: str) -> str:
    """Search the web for a query. Returns top results with titles, URLs, and snippets."""
    print(f'\n🔍 Searching: {query}')
    results = []
    for r in DDGS().text(query, max_results=4):
        results.append(f"Title: {r['title']}\nURL: {r['href']}\nSnippet: {r['body']}")
    return '\n\n'.join(results) if results else 'No results found.'


@tool
def open_and_read(url: str) -> str:
    """Open a URL in the browser AND read its text content. Use this to get full details from a webpage."""
    print(f'\n🌐 Opening browser: {url}')
    webbrowser.open(url)  # Opens the actual browser window!

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        return text[:3000] + ('...' if len(text) > 3000 else '')
    except Exception as e:
        return f'Could not read page: {e}'


tools = [search_web, open_and_read]
print('✅ 2 tools ready!')
print('  🔍 search_web   — searches DuckDuckGo')
print('  🌐 open_and_read — opens browser + reads page')

✅ 2 tools ready!
  🔍 search_web   — searches DuckDuckGo
  🌐 open_and_read — opens browser + reads page


In [13]:
agent = create_agent(llm, tools)
print('✅ Browser Agent is ready!')

✅ Browser Agent is ready!


In [14]:
def ask(question):
    print('=' * 60)
    print(f'❓ QUESTION: {question}')
    print('=' * 60)
    result = agent.invoke({'messages': [('user', question)]})
    answer = result['messages'][-1].content
    print('\n' + '=' * 60)
    print('✅ ANSWER:')
    print('=' * 60)
    print(answer)
    return answer

In [15]:
# Demo 1 — Current news
ask('What is the latest news about artificial intelligence today?')

❓ QUESTION: What is the latest news about artificial intelligence today?

🔍 Searching: latest news artificial intelligence today

🌐 Opening browser: https://news.google.com/topics/CAAqJAgKIh5DQkFTRUFvSEwyMHZNRzFyZWhJRlpXNHRSMElvQUFQAQ

✅ ANSWER:
Based on the search results, it appears that there are several recent developments and news articles related to artificial intelligence. Some of the key points include:

* OpenAI's CEO Aravind Srinivas has stated that computer science is gradually returning to the domain of art.
* There have been reports of AI-generated content being used to create "sexy suicide coach" videos.
* Israeli Prime Minister Netanyahu's coffee shop video has been claimed to be AI-generated, fueling rumors about his death.
* Aman Gottumukkala, an Indian-origin founder, is joining xAI after building a million-dollar startup with just Rs 5 lakh.
* AI-powered agents are being used to make significant profits in the stock market.
* Dubai is deploying AI-powered drones and 

'Based on the search results, it appears that there are several recent developments and news articles related to artificial intelligence. Some of the key points include:\n\n* OpenAI\'s CEO Aravind Srinivas has stated that computer science is gradually returning to the domain of art.\n* There have been reports of AI-generated content being used to create "sexy suicide coach" videos.\n* Israeli Prime Minister Netanyahu\'s coffee shop video has been claimed to be AI-generated, fueling rumors about his death.\n* Aman Gottumukkala, an Indian-origin founder, is joining xAI after building a million-dollar startup with just Rs 5 lakh.\n* AI-powered agents are being used to make significant profits in the stock market.\n* Dubai is deploying AI-powered drones and aquatic rescue bots on beaches to enhance coastal security.\n* NVIDIA is facing competition from Google and Meta in the AI market.\n* Accenture\'s chief Julia Sweet has warned of job loss and laid out conditions for promotion.\n* Robots

In [16]:
# Demo 2 — Tech info
ask('What is the latest version of Python and what are its new features?')

❓ QUESTION: What is the latest version of Python and what are its new features?

🔍 Searching: latest version of Python and its new features

🌐 Opening browser: https://www.python.org/downloads/

🌐 Opening browser: https://docs.python.org/3/whatsnew/

✅ ANSWER:
The latest version of Python is Python 3.14, which was released on October 7, 2025. Some of the new features in Python 3.14 include:

* Free-threaded mode
* JIT compiler
* Lazy annotations
* Modern colorized REPL

Python 3.14 also includes improvements to exception-handling, a new approach to string templating, and improvements to the use of concurrent interpreters.

It's worth noting that Python 3.14 is the latest stable release of the Python programming language, and it's recommended to upgrade to this version for the latest features and security patches.

Additionally, Python 3.14 has a mix of changes to the language, the implementation, and the standard library, which can be found in the Python 3.14 documentation.


"The latest version of Python is Python 3.14, which was released on October 7, 2025. Some of the new features in Python 3.14 include:\n\n* Free-threaded mode\n* JIT compiler\n* Lazy annotations\n* Modern colorized REPL\n\nPython 3.14 also includes improvements to exception-handling, a new approach to string templating, and improvements to the use of concurrent interpreters.\n\nIt's worth noting that Python 3.14 is the latest stable release of the Python programming language, and it's recommended to upgrade to this version for the latest features and security patches.\n\nAdditionally, Python 3.14 has a mix of changes to the language, the implementation, and the standard library, which can be found in the Python 3.14 documentation."

In [17]:
# Demo 3 — Cricket
ask('Who won the most recent cricket World Cup and what was the score?')

❓ QUESTION: Who won the most recent cricket World Cup and what was the score?

🔍 Searching: most recent cricket World Cup winner and score

🌐 Opening browser: https://en.wikipedia.org/wiki/Cricket_World_Cup

✅ ANSWER:
The most recent cricket World Cup was won by Australia, and the score is not specified in the text. However, the text mentions that Australia has won the tournament six times, and the most recent win was in 2023.


'The most recent cricket World Cup was won by Australia, and the score is not specified in the text. However, the text mentions that Australia has won the tournament six times, and the most recent win was in 2023.'

---
## 🎮 YOUR TURN — Ask Anything!
> Change `my_question` and run. Watch the browser open!

In [18]:
# Type YOUR question here!
my_question = 'What is LangGraph and how is it different from LangChain?'

ask(my_question)

❓ QUESTION: What is LangGraph and how is it different from LangChain?

🔍 Searching: LangGraph vs LangChain

🌐 Opening browser: https://langchain.dev/

🌐 Opening browser: https://langgraph.com/

✅ ANSWER:
Based on the information provided, LangChain and LangGraph are both AI-related technologies, but they have different focuses and architectures.

LangChain is a platform for building and deploying AI agents, with a focus on chaining LLM (Large Language Model) operations in a specific sequence. It provides a range of tools and features for building, testing, and deploying agents, including observability, evaluation, and deployment capabilities.

LangGraph, on the other hand, is a graph-based architecture that builds on top of LangChain. It is designed for creating stateful, multi-agent systems, and is better suited for complex decision-making and stateful applications. LangGraph provides a more advanced and scalable architecture than LangChain, but requires more upfront effort to set up 

'Based on the information provided, LangChain and LangGraph are both AI-related technologies, but they have different focuses and architectures.\n\nLangChain is a platform for building and deploying AI agents, with a focus on chaining LLM (Large Language Model) operations in a specific sequence. It provides a range of tools and features for building, testing, and deploying agents, including observability, evaluation, and deployment capabilities.\n\nLangGraph, on the other hand, is a graph-based architecture that builds on top of LangChain. It is designed for creating stateful, multi-agent systems, and is better suited for complex decision-making and stateful applications. LangGraph provides a more advanced and scalable architecture than LangChain, but requires more upfront effort to set up and use.\n\nIn summary, LangChain is a more general-purpose platform for building and deploying AI agents, while LangGraph is a more specialized architecture for building complex, stateful systems. T

---
## How It Works

```
ask("latest Python version?")
  → Agent: search_web("latest Python version")
  → Gets URLs from DuckDuckGo
  → Agent: open_and_read("https://python.org/...")
  → Browser opens  ←  you see this!
  → Page text scraped silently
  → LLM reads text → answers
  → ✅ "Python 3.13 was released in..."
```

| Code | What it does |
|---|---|
| `@tool` | Makes any Python function usable by the agent |
| `webbrowser.open()` | Opens your default browser (built-in Python) |
| `BeautifulSoup` | Strips HTML, returns clean readable text |
| `create_react_agent` | Builds the Think → Act → Observe loop |

*iTech Academy | AI Workshop | Browser Agent Demo*